In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError
 
from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)
 
def get_aws_credentials(secrets_scope, key_name):
    try:  
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
 
def get_scope_key_names(secrets_scope):
    try:
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
    return response['Parameter']['Value']
 
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    aws_secret_key = get_aws_credentials(secrets_scope, aws_key)
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
    env = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env

env = get_env_prefix()

run_date_param = dbutils.widgets.get("run_date").strip()

if run_date_param:
    ref_date = run_date_param
else:
    ref_date = str(spark.sql(f"""
        SELECT MAX(BusinessDate) AS bd FROM `{env}-cdh`.derivativeactive.dexposure
    """).collect()[0].bd)

target_schema = f"`{env}-cdh-delivery`.it"

print(f"ref_date = {ref_date}")
print(f"env = {env}")

In [0]:
#env = 'p'
# target_schema = f"`d-cdh-delivery`.it"

In [0]:
# =============================================================================
# SOURCE TABLE VIEWS
# =============================================================================

# --- 1. Payment Schedule (primary, at ref_date, profits only) ---
# ExposureKey IS NOT NULL: excludes orphaned rows with no parent loan
# All amount columns cast to DECIMAL(19,2) for consistent schema after SUM
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW paymentschedule AS
    SELECT
        ExposureKey,
        PortfolioKey,
        InstallmentDate,
        InstallmentTypeCode,
        InstallmentNumber,
        CAST(InstallmentAmount AS DECIMAL(19,2)) AS InstallmentAmount,
        CAST(CapitalAmount AS DECIMAL(19,2)) AS CapitalAmount,
        CAST(InterestAmount AS DECIMAL(19,2)) AS InterestAmount
    FROM `{env}-cdh`.primaryactive.paymentschedule
    WHERE BusinessDate = '{ref_date}'
      AND SourceSystem = 'profits'
      AND ExposureKey IS NOT NULL
""")

# --- 2. Exposures via function (each account at its latest available date) ---
# HasPaymentPlanFlag: used to determine NULL TotalContractualInstalments
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW dexposure AS
    SELECT
        ExposureKey,
        PortfolioKey,
        IsActiveFlag,
        HasActiveSettlementFlag,
        HasPaymentPlanFlag
    FROM `{env}-cdh`.derivative.get_exposures_by_business_date(DATE'{ref_date}', true)
    WHERE SourceSystem = 'profits'
""")

# --- 3. Collections at ref_date (payments only) ---
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW dcollection AS
    SELECT
        ExposureKey,
        ActualDate,
        CAST(Amount AS DECIMAL(19,2)) AS Amount
    FROM `{env}-cdh`.derivativeactive.dcollection
    WHERE BusinessDate = '{ref_date}'
      AND IsPaymentFlag = true
""")

# --- 4. Portfolio mapping: unit_code_profits (Profits key) → PortfolioId (on-prem DWH code) ---
# unit_code_profits = Profits PortfolioKey (e.g. 35)
# PortfolioId       = DWH portfolio ID used by on-prem OperationEsbPaymentPlan (e.g. 82)
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW portfolio AS
    SELECT DISTINCT
        unit_code_profits,
        PortfolioId
    FROM `{env}-cdh`.primaryactive.portfolio
    WHERE unit_code_profits IS NOT NULL
      AND BusinessDate = (SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.portfolio)
""")

In [0]:
# =============================================================================
# ESB Payment Plan - Primary Source
# =============================================================================

print(f"Building ESB_PaymentPlan (ref_date={ref_date}, env={env})")
print(f"Target: {target_schema}.esb_paymentplan")

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW v_esb_primary AS
    WITH deduped AS (
        -- Step 1: Remove true duplicates (same key + same amounts)
        SELECT DISTINCT
            ExposureKey,
            PortfolioKey,
            InstallmentDate,
            InstallmentTypeCode,
            InstallmentNumber,
            InstallmentAmount,
            CapitalAmount,
            InterestAmount
        FROM paymentschedule
    ),
    aggregated AS (
        -- Step 2: Consolidate multiple PaymentScheduleIds per installment
        SELECT
            ExposureKey,
            MAX(PortfolioKey) AS PortfolioKey,
            InstallmentDate,
            InstallmentTypeCode,
            MAX(InstallmentNumber) AS InstallmentNumber,
            CAST(SUM(InstallmentAmount) AS DECIMAL(19,2)) AS InstallmentAmount,
            CAST(SUM(CapitalAmount) AS DECIMAL(19,2)) AS CapitalAmount,
            CAST(SUM(InterestAmount) AS DECIMAL(19,2)) AS InterestAmount
        FROM deduped
        GROUP BY ExposureKey, InstallmentDate, InstallmentTypeCode
    )
    SELECT
        p.ExposureKey                       AS AccountNumber
       ,COALESCE(CAST(port.PortfolioId AS STRING), COALESCE(e.PortfolioKey, CAST(p.PortfolioKey AS STRING)))  AS PortfolioId
       ,CAST(p.InstallmentDate AS DATE)     AS InstallmentDate
       ,p.InstallmentAmount
       ,CAST(ROW_NUMBER() OVER (
            PARTITION BY p.ExposureKey
            ORDER BY p.InstallmentDate, p.InstallmentTypeCode
        ) AS INT)                           AS InstallmentNumber
       ,p.CapitalAmount                     AS Capital
       ,p.InterestAmount                    AS Interest
       ,CAST(CASE
            WHEN e.HasPaymentPlanFlag = false AND e.IsActiveFlag = false THEN NULL
            ELSE MAX(p.InstallmentNumber) OVER (PARTITION BY p.ExposureKey)
        END AS INT)                         AS TotalContractualInstalments
       ,CAST(c.Amount AS DECIMAL(19,2))     AS AmountPaid
       ,CAST(c.ActualDate AS DATE)          AS DateOfPayment
       ,CASE
            WHEN p.InstallmentTypeCode IN ('1','2','4','5') THEN p.InstallmentTypeCode
            ELSE '1'
        END                                 AS RequestType
       ,p.ExposureKey                       AS LoanId
       ,DATE'{ref_date}'                    AS DataDate
       ,CAST(e.IsActiveFlag AS INT)         AS IsLoanActiveFlag
       ,CASE
            WHEN e.HasActiveSettlementFlag = true AND e.IsActiveFlag = true
            THEN 1 ELSE 0
        END                                 AS IsRunning

    FROM aggregated p

    LEFT JOIN dexposure e
      ON e.ExposureKey = p.ExposureKey

    LEFT JOIN portfolio port
      ON port.unit_code_profits = CAST(COALESCE(e.PortfolioKey, p.PortfolioKey) AS INT)

    LEFT JOIN dcollection c
      ON  c.ExposureKey  = p.ExposureKey
      AND MONTH(c.ActualDate) = MONTH(p.InstallmentDate)
      AND YEAR(c.ActualDate)  = YEAR(p.InstallmentDate)
""")

# --- Write to production target ---
result_df = spark.sql("SELECT * FROM v_esb_primary")

result_df.write.format("delta") \
    .mode("overwrite") \
    .option("encoding", "UTF-8") \
    .saveAsTable(f"{target_schema}.esb_paymentplan")